In [1]:
import time # Added for measuring execution time

def get_moves(state):
    """Generates all valid next states from the current board state."""
    zero_idx = state.index(0)
    row, col = divmod(zero_idx, 3)
    moves = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right

    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            state_list = list(state)
            state_list[zero_idx], state_list[new_idx] = state_list[new_idx], state_list[zero_idx]
            moves.append(tuple(state_list))
    return moves

def depth_limited_dfs(current, goal, limit, path, visited, nodes_explored):
    """A recursive, depth-limited DFS helper function."""
    nodes_explored[0] += 1 # Increment node count for each state visited

    if current == goal:
        return path
    if limit <= 0:
        return None

    visited.add(current)
    for next_state in get_moves(current):
        if next_state not in visited:
            result = depth_limited_dfs(next_state, goal, limit - 1, path + [next_state], visited, nodes_explored)
            if result is not None:
                return result
    visited.remove(current)  # Backtracking keeps space bounded to path length
    return None

def iddfs(start, goal, max_depth_limit=None):
    """Main IDDFS loop that increments the depth limit progressively."""
    depth = 0
    total_nodes_explored = [0] # Using a list to pass by reference for mutable update

    while True:
        print(f"Searching at depth limit: {depth}...")
        visited = set()
        nodes_explored_this_depth = [0] # Count for the current DLS iteration
        result = depth_limited_dfs(start, goal, depth, [start], visited, nodes_explored_this_depth)
        total_nodes_explored[0] += nodes_explored_this_depth[0]

        if result is not None:
            return result, depth, total_nodes_explored[0]
        depth += 1
        # Stop if a maximum depth limit is provided and reached
        if max_depth_limit is not None and depth > max_depth_limit:
            print(f"Max depth limit of {max_depth_limit} reached. No solution found within this limit.")
            return None, max_depth_limit, total_nodes_explored[0]

def print_board(state):
    """Prints a 1D tuple formatted beautifully as a 3x3 board."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]}")
    print()

def get_user_input_board(prompt_name):
    """Prompts the user for a valid 8-puzzle board configuration row by row."""
    while True:
        print(f"\nEnter the {prompt_name} state row by row.")
        print("Provide 3 numbers per row separated by spaces. Use '0' for the blank space.")

        digits = []
        error_occurred = False

        for r in range(3):
            try:
                row_str = input(f"Row {r+1}: ").strip()
                row_digits = [int(x) for x in row_str.split()]
                if len(row_digits) != 3:
                    print("❌ Error: Each row must contain exactly 3 numbers.")
                    error_occurred = True
                    break
                digits.extend(row_digits)
            except ValueError:
                print("❌ Error: Invalid characters. Please enter numbers only.")
                error_occurred = True
                break

        if error_occurred:
            continue

        # Validation checks for the complete board
        if len(digits) != 9 or set(digits) != set(range(9)):
            print("❌ Error: The complete board must contain exactly numbers from 0 to 8 with no duplicates.")
            continue

        return tuple(digits)

def get_user_input_int(prompt):
    """Prompts the user for a valid integer input."""
    while True:
        try:
            value = int(input(prompt).strip())
            if value < 0:
                print("❌ Error: Please enter a non-negative integer.")
            else:
                return value
        except ValueError:
            print(" Error: Invalid input. Please enter a whole number.")

# --- Main Driver Script ---
if __name__ == "__main__":
    print("=== 8-Puzzle Solver using Iterative Deepening DFS (IDDFS) ===")
    start_state = get_user_input_board("STARTING")
    goal_state = get_user_input_board("GOAL")
    max_depth_limit = get_user_input_int("\nEnter a maximum depth limit for the search (e.g., 20 for a reasonable cap, 0 for no limit): ")

    # If the user enters 0 for max_depth_limit, treat it as no limit
    if max_depth_limit == 0:
        max_depth_limit = None

    print("\n--- Initializing Search ---")
    print("Initial State:")
    print_board(start_state)
    print("Goal State:")
    print_board(goal_state)
    if max_depth_limit is not None:
        print(f"Max Depth Limit: {max_depth_limit}")

    start_timer = time.time() # Start timer
    solution_path, final_depth, explored_nodes_count = iddfs(start_state, goal_state, max_depth_limit)
    end_timer = time.time() # End timer
    execution_time = end_timer - start_timer

    if solution_path is not None:
        print(f"\n✨ Success! Optimal solution found at depth: {final_depth}")
        print(f"Total moves required: {len(solution_path) - 1}")
        print(f"Total nodes explored: {explored_nodes_count}")

        print("\n--- Step-by-Step Path ---")
        for step, state in enumerate(solution_path):
            if step == 0:
                print(f"Step {step} (Start):")
            else:
                print(f"Step {step}:")
            print_board(state)

        # --- Empirical Performance for IDDFS ---
        print(f"\n--- Empirical Performance (IDDFS) ---")
        print(f"Total Explored States (Nodes processed across all DLS iterations): {explored_nodes_count}")
        print(f"Execution Time: {execution_time:.4f} seconds")
        print(f"Max Stack Depth for a DLS iteration (path length): {final_depth + 1} nodes")

        # --- Theoretical Complexities for IDDFS ---
        branching_factor = 3 # For 8-puzzle, average branching factor is 2-4. Using 3 as a representative value.
        theoretical_time_nodes_approx = branching_factor ** final_depth
        theoretical_space_nodes = branching_factor ** final_depth

        print(f"\n--- Theoretical Complexities (IDDFS) ---")
        print(f"For Solution Depth (d) = {final_depth}, Branching Factor (b) = {branching_factor}")
        print(f"Theoretical Time Complexity (Nodes explored, approximated O(b^d)): {int(theoretical_time_nodes_approx)} nodes (note: IDDFS explores nodes at shallower depths multiple times)")
        print(f"Theoretical Space Complexity (Max stack depth * branching factor, O(b*d)): {int(theoretical_space_nodes)} nodes")

    else:
        print(f"\nNo solution found within the specified maximum depth limit of {max_depth_limit}.")
        print(f"Total nodes explored until limit: {explored_nodes_count}")

        # Add empirical performance even if no solution
        print(f"\n--- Empirical Performance (IDDFS until limit) ---")
        print(f"Total Explored States (Nodes processed across all DLS iterations): {explored_nodes_count}")
        print(f"Execution Time: {execution_time:.4f} seconds")
        if max_depth_limit is not None:
             print(f"Max Stack Depth for a DLS iteration (path length): {max_depth_limit + 1} nodes (reached at max depth limit)")
        else:
             print(f"Max Stack Depth for a DLS iteration (path length): Unknown (no limit specified, search exhausted)")

    print("\n--- General Complexity Analysis for Iterative Deepening DFS (IDDFS) ---")
    print("Time Complexity: O(b^d)")
    print("  'b' is the branching factor (average number of possible moves from a state). For the 8-puzzle, b is typically around 2-4.")
    print("  'd' is the depth of the shallowest goal state. IDDFS explores states up to depth 'd' multiple times.")
    print("Space Complexity: O(b^d)")
    print("  IDDFS performs a Depth-First Search at each iteration, so its space complexity is determined by the maximum depth of the DFS stack, which is 'd'.")

=== 8-Puzzle Solver using Iterative Deepening DFS (IDDFS) ===

Enter the STARTING state row by row.
Provide 3 numbers per row separated by spaces. Use '0' for the blank space.
Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 7 0 5

Enter the GOAL state row by row.
Provide 3 numbers per row separated by spaces. Use '0' for the blank space.
Row 1: 1 2 3
Row 2: 8 0 4
Row 3: 7 6 5

Enter a maximum depth limit for the search (e.g., 20 for a reasonable cap, 0 for no limit): 5

--- Initializing Search ---
Initial State:
 2 8 3
 1 6 4
 7 0 5

Goal State:
 1 2 3
 8 0 4
 7 6 5

Max Depth Limit: 5
Searching at depth limit: 0...
Searching at depth limit: 1...
Searching at depth limit: 2...
Searching at depth limit: 3...
Searching at depth limit: 4...
Searching at depth limit: 5...

✨ Success! Optimal solution found at depth: 5
Total moves required: 5
Total nodes explored: 73

--- Step-by-Step Path ---
Step 0 (Start):
 2 8 3
 1 6 4
 7 0 5

Step 1:
 2 8 3
 1 0 4
 7 6 5

Step 2:
 2 0 3
 1 8 4
 7 6 5

Step 3:
 0 2 3
